# DATA 266 Lab 2 - Part 1: Hybrid Multimodal RAG baseline

This notebook runs the reproducible ingestion, retrieval, submission generation, and schema validation path.

In [ ]:
from pathlib import Path
import sys, subprocess, platform
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'part1_rag').is_dir(): ROOT = ROOT.parent
PART1 = ROOT / 'part1_rag'
sys.path.insert(0, str(PART1 / 'src'))
print('repository:', ROOT)
print('python:', platform.python_version())
try:
    import torch
    print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('gpu:', torch.cuda.get_device_name(0))
except Exception as e: print('torch unavailable:', e)

In [ ]:
from rag_pipeline import build_index
index = build_index(PART1 / 'data')
print('indexed passages:', len(index.passages))
for hit in index.search('What happened to unemployment after the crisis?', k=3):
    print(round(hit['score'], 4), hit['passage']['doc_id'], hit['passage']['page'])

In [ ]:
out = PART1 / 'outputs' / 'submission.csv'
subprocess.run([sys.executable, str(PART1 / 'src' / 'generate_submission.py'), '--data-dir', str(PART1 / 'data'), '--output', str(out)], check=True)
subprocess.run([sys.executable, str(PART1 / 'data' / 'validate_submission.py'), str(out), str(PART1 / 'data' / 'Questions.csv')], check=True)
print('submission:', out)